[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BioProteanLabs/Tutorials/blob/main/Glowstick_Chemiluminescence_Tutorial.ipynb)

# Tutorial: fitting glowstick chemiluminescence

This example fits the measured light intensity from `200_83_85_85.csv` with a second-order kinetic model. Column 1 contains time in milliseconds, column 2 contains measured intensity, and the remaining columns are sliding averages that are not used.

For a second-order reaction in one reactant, `[A](t) = [A]₀ / (1 + k₂[A]₀t)`. If light intensity is proportional to reaction rate, the intensity has a squared-decay form: `I(t) = I_background + I₀_excess / (1 + k_eff t)²`, where `k_eff = k₂[A]₀`. The intensity fit estimates `k_eff`; given `[A]₀`, we can then calculate `k₂`.

The photon-emission example also needs the reaction volume and quantum yield. The values below are examples; replace them with measured or documented conditions for absolute photon rates.

In [ ]:
# Libraries used throughout the tutorial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

%matplotlib inline
plt.style.use('dark_background')

In [ ]:
def second_order_intensity(time_s, background, initial_excess, k_eff, *, observed=None):
    """Predict intensity from a second-order reaction-rate model.

    If observed intensities are provided, also return the residual variance
    corrected for the three fitted parameters.
    """
    fitted = background + initial_excess / (1 + k_eff * time_s) ** 2
    if observed is None:
        return fitted

    residual = np.asarray(observed) - fitted
    degrees_of_freedom = residual.size - 3
    if degrees_of_freedom <= 0:
        raise ValueError('Need more observations than fitted parameters to estimate residual variance.')
    residual_variance = np.sum(residual ** 2) / degrees_of_freedom
    return fitted, residual_variance


def fit_second_order_intensity(time_s, intensity):
    """Fit the second-order intensity model and summarize residuals."""
    time_s = np.asarray(time_s, dtype=float)
    intensity = np.asarray(intensity, dtype=float)

    initial_guess = (
        float(intensity.min()),
        float(intensity.max() - intensity.min()),
        1 / 500,
    )

    # A wrapper keeps the optimizer's model signature to the three fit parameters.
    def model_for_curve_fit(time, background, initial_excess, k_eff):
        return second_order_intensity(time, background, initial_excess, k_eff)

    parameters, covariance = curve_fit(
        model_for_curve_fit,
        time_s,
        intensity,
        p0=initial_guess,
        bounds=([-np.inf, 0, 0], [np.inf, np.inf, np.inf]),
        maxfev=20_000,
    )
    fitted, residual_variance = second_order_intensity(
        time_s, *parameters, observed=intensity
    )
    residuals = intensity - fitted
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((intensity - intensity.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot

    return {
        'parameters': parameters,
        'standard_errors': np.sqrt(np.diag(covariance)),
        'fitted': fitted,
        'residuals': residuals,
        'residual_variance': residual_variance,
        'r_squared': r_squared,
    }


def second_order_concentration(time_s, initial_concentration, k2):
    """Return [A](t) for a second-order reaction in one reactant."""
    return initial_concentration / (1 + k2 * initial_concentration * time_s)


def calculate_photon_emission(t, A0, k, volume_ml, quantum_yield):
    """Return molecules decaying/s and photons emitted/s at time t."""
    if volume_ml <= 0:
        raise ValueError('volume_ml must be positive.')
    if not 0 <= quantum_yield <= 1:
        raise ValueError('quantum_yield must be between 0 and 1.')

    avogadro_constant = 6.022e23  # molecules/mol
    volume_l = volume_ml / 1000
    rate_mol_per_l_s = (k * A0**2) / (1 + k * A0 * t) ** 2
    molecules_per_s = rate_mol_per_l_s * volume_l * avogadro_constant
    photons_per_s = molecules_per_s * quantum_yield
    return molecules_per_s, photons_per_s

## Load and inspect the CSV

The file wraps each comma-separated row in one quoted CSV field and ends with `TEST_COMPLETE`. We split each row, discard the nonnumeric footer, and retain only the time and measured-intensity columns. In Colab, upload `200_83_85_85.csv` when prompted.

In [ ]:
possible_paths = [
    Path('pharmacology/200_83_85_85.csv'),  # run from the project root
    Path('../pharmacology/200_83_85_85.csv'),  # run from Tutorials/
]
csv_path = next((path for path in possible_paths if path.exists()), None)
if csv_path is None and IN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Upload 200_83_85_85.csv to continue.')
    csv_path = Path(next(iter(uploaded)))
elif csv_path is None:
    raise FileNotFoundError('Could not find pharmacology/200_83_85_85.csv from the current working directory.')

raw = pd.read_csv(csv_path, header=None, names=['record'], dtype=str)
split_columns = raw['record'].str.split(',', expand=True)
split_columns = split_columns.apply(pd.to_numeric, errors='coerce')
numeric_rows = split_columns.dropna(subset=[0, 1])
data = numeric_rows[[0, 1]].copy()
data.columns = ['time_ms', 'intensity']
data = data.sort_values('time_ms').reset_index(drop=True)
time_s = data['time_ms'].to_numpy(dtype=float) / 1000
intensity = data['intensity'].to_numpy(dtype=float)

print(f'Loaded {len(data):,} observations from {csv_path}')
display(data.head())
display(data.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(time_s, intensity, s=8, alpha=0.4, label='measured intensity')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Light intensity (instrument units)')
ax.set_title('Glowstick chemiluminescence measurements')
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Fit the intensity data

The fit estimates background intensity, initial intensity above background, and the effective decay rate `k_eff`. Residual variance is the sum of squared residuals divided by `n − 3`, accounting for the three fitted parameters.

In [ ]:
fit = fit_second_order_intensity(time_s, intensity)
background, initial_excess, k_eff = fit['parameters']
background_se, initial_excess_se, k_eff_se = fit['standard_errors']
fitted_intensity = fit['fitted']
residuals = fit['residuals']

print(f'Background intensity: {background:.4g} ± {background_se:.2g}')
print(f'Initial excess intensity: {initial_excess:.4g} ± {initial_excess_se:.2g}')
print(f'Effective decay rate, k_eff: {k_eff:.4g} ± {k_eff_se:.2g} s⁻¹')
print(f'Residual variance: {fit["residual_variance"]:.4g} intensity units²')
print(f'R²: {fit["r_squared"]:.5f}')

In [ ]:
time_curve = np.linspace(time_s.min(), time_s.max(), 800)
fitted_curve = second_order_intensity(time_curve, background, initial_excess, k_eff)
fig, (ax_fit, ax_residual) = plt.subplots(2, 1, figsize=(11, 8), sharex=True,
                                           gridspec_kw={'height_ratios': [3, 1]})
ax_fit.scatter(time_s, intensity, s=8, alpha=0.3, label='measured intensity')
ax_fit.plot(time_curve, fitted_curve, color='tomato', linewidth=2, label='second-order fit')
ax_fit.set_ylabel('Light intensity (instrument units)')
ax_fit.set_title(f'Second-order fit (R² = {fit["r_squared"]:.4f})')
ax_fit.grid(True, alpha=0.25)
ax_fit.legend()
ax_residual.scatter(time_s, residuals, s=6, alpha=0.35)
ax_residual.axhline(0, color='tomato', linewidth=1)
ax_residual.set_xlabel('Time (s)')
ax_residual.set_ylabel('Residual')
ax_residual.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Estimate concentration over time

For this worked example, the starting concentration is 3.77 mol/L. Since `k_eff = k₂[A]₀`, calculate `k₂ = k_eff / [A]₀`, then use the integrated second-order concentration equation.

In [ ]:
starting_concentration = 3.77  # mol/L
k2 = k_eff / starting_concentration  # L/(mol·s)
concentration_time = np.linspace(0, time_s.max(), 800)
concentration = second_order_concentration(concentration_time, starting_concentration, k2)
print(f'Estimated k₂: {k2:.4g} L/(mol·s)')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(concentration_time, concentration, color='mediumseagreen', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Concentration (mol/L)')
ax.set_title('Second-order concentration curve')
ax.grid(True, alpha=0.25)
plt.show()

## Worked example: estimate photon emission

The function returns molecules decaying per second and photons emitted per second. Here we use the fitted kinetics with example conditions of 2.0 mL reaction volume and 0.15 quantum yield (15%). Use the experiment's actual volume and quantum yield when known.

In [ ]:
volume_ml = 2.0  # example reaction volume
quantum_yield = 0.15  # example: 15% of reacted molecules emit a photon

example_time_s = 5.0
molecules_per_s, photons_per_s = calculate_photon_emission(
    t=example_time_s,
    A0=starting_concentration,
    k=k2,
    volume_ml=volume_ml,
    quantum_yield=quantum_yield,
)
print(f'At t = {example_time_s:g} s:')
print(f'  Molecules decaying per second: {molecules_per_s:.3e}')
print(f'  Photons emitted per second: {photons_per_s:.3e}')

In [ ]:
last_interval_s = int(time_s.max() // 500) * 500
interval_seconds = np.arange(0, last_interval_s + 1, 500)
emission = [
    calculate_photon_emission(
        t=float(t), A0=starting_concentration, k=k2,
        volume_ml=volume_ml, quantum_yield=quantum_yield
    )
    for t in interval_seconds
]
photon_rates = np.array([result[1] for result in emission])
photon_table = pd.DataFrame({
    'time_s': interval_seconds,
    'molecules_decaying_per_s': [result[0] for result in emission],
    'photons_emitted_per_s': photon_rates,
})
display(photon_table)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(interval_seconds, photon_rates, marker='o', color='gold', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Photons emitted per second')
ax.set_title('Estimated photon emission every 500 seconds')
ax.grid(True, alpha=0.25)
plt.show()